In [ ]:
import numpy as np
import h5py
import matplotlib.pyplot as plt

from skimage.measure import block_reduce
from helper_functions import write_text

In [ ]:
# Loading strict AGIPD mask
det_mask_file = 'emc/make_detector/full_mask_agipd_ds_4_strict_mask.h5'
with h5py.File(det_mask_file, 'r') as det:
    det_mask_ds = det['mask'][:]

In [ ]:
dsf = 4
num_runs = 50
n_sim = 2000
def_rng = np.random.default_rng()

for nr in range(num_runs):
    write_text(f'\rResampling run {nr+1}/{num_runs}...\n')
    # Loading continuous scattering pattern and setting resampling factor
    run_file = f'sims_protein_water/run_{nr}_protein_in_water_100k_pats_dsf_{dsf}x/particle_intens.npy'
    particle_intens = np.load(run_file)[:]
    
    det_mask_ds_stack = np.broadcast_to(det_mask_ds, (n_sim,) + det_mask_ds.shape).astype(np.float64)
    particle_intens[det_mask_ds_stack==False] = 0.0

    # Poisson sampling continuous protein scattering pattern
    poiss_samp_remasked = def_rng.poisson(lam=particle_intens)

    # Saving remasking diffraction patterns
    file_name_poiss_samp = f'sims_protein_water/run_{nr}_protein_in_water_100k_pats_dsf_{dsf}x/poisson_prot_strict_remask.npy'
    
    print("Saving remasked arrays...")
    np.save(file_name_poiss_samp, arr=poiss_samp_remasked)
    print("Finished saving remasked arrays...")

write_text('All remasking done...')